# Pandas index objects

**Learning Objectives:**
- Understand what index objects are and why they matter in pandas
- Work with different index types: `RangeIndex`, label-based `Index`, and `MultiIndex`
- Use `loc` and `iloc` to access data with different index types
- Set and reset DataFrame indices effectively
- Navigate hierarchical data with `MultiIndex`

**Requirements:** Python 3.11+, pandas 2.0+

---

## 1. What are index objects?

📚 **Concept:** Every pandas DataFrame has an **index** - it's like a row label that helps you locate and organize your data. Think of it like page numbers in a book or row headers in a spreadsheet.

There are three main types of indices we'll cover:
1. **RangeIndex** - Default numeric index (0, 1, 2, ...)
2. **Index** - Custom labels (names, dates, IDs, etc.)
3. **MultiIndex** - Multiple levels of labels (like nested categories)

<!-- INSTRUCTOR: Emphasize that students have already been using indices without realizing it - the numbers on the left side of every DataFrame they've seen -->

In [ ]:
import pandas as pd

---

## 2. RangeIndex (the default)

📚 **Concept:** When you load a CSV file or create a DataFrame, pandas automatically creates a `RangeIndex` - a sequence of integers starting from 0.

In [ ]:
# Load the Titanic dataset
df = pd.read_csv("titanic.csv")

In [ ]:
# View the first few rows
df.head()

💡 **Tip:** Notice the numbers 0, 1, 2, 3, 4 on the left? That's the index!

In [1]:
# Access the index directly
df.index

NameError: name 'df' is not defined

In [ ]:
# Check the type of index
type(df.index)

📚 **Concept:** The `RangeIndex` is memory-efficient - it doesn't store all numbers, just the start (0), stop (891), and step (1).

In [ ]:
# See first 10 index values
df.index.values[:10]

### 2.1 Resetting the index

📚 **Concept:** Sometimes operations (like filtering or sorting) can make your index "messy" with gaps or non-sequential numbers. The `.reset_index()` method creates a clean new `RangeIndex`.

In [ ]:
# Check data info
df.info()

In [ ]:
# Remove rows with any missing values
df_clean = df.dropna()
df_clean.info()

💡 **Tip:** Notice the index still preserves the original row numbers after filtering? The index type may show as `Index` (in pandas 2.0+) instead of the old `RangeIndex` - this is normal when rows are filtered.

In [ ]:
# See the "messy" index with gaps
df_clean.head(10)

In [ ]:
# Reset to create a clean 0, 1, 2... sequence
df_reset = df_clean.reset_index()
df_reset.head()

⚠️ **Warning:** By default, `.reset_index()` saves the old index as a new column called `index`. Usually you don't want this!

In [ ]:
# Use drop=True to discard the old index
df_reset = df_clean.reset_index(drop=True)
df_reset.head()

💡 **Tip:** Modern pandas style uses method chaining instead of `inplace=True`. This makes code more readable and safer:

```python
# Modern approach (recommended)
df_clean = df.dropna().reset_index(drop=True)

# Older approach (avoid when possible)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
```

✅ **Key Takeaway:** Use `.reset_index(drop=True)` to create a clean sequential index after filtering or sorting data.

---

## 3. Understanding `loc` vs `iloc`

📚 **Concept:** There are two ways to select rows in pandas:
- `.loc[]` - Select by **label** (index name)
- `.iloc[]` - Select by **integer location** (position number)

With a `RangeIndex`, these work the same way because labels ARE integers. But this changes with custom indices!

In [ ]:
# View first row to compare
df_reset.head(1)

In [ ]:
# Locate row with label 0
df_reset.loc[0]

In [ ]:
# Locate row at position 0 (first row)
df_reset.iloc[0]

In [ ]:
# They're identical with RangeIndex
df_reset.loc[0].equals(df_reset.iloc[0])

### 3.1 Selecting rows and columns together

💡 **Tip:** You can select specific rows and columns in one operation using `[row, column]` syntax.

In [ ]:
# Get the fare value from the first row using loc
df_reset.loc[0, "fare"]

In [ ]:
# Same thing using iloc (fare is the 7th column, index 6)
df_reset.iloc[0, 6]

⚠️ **Warning:** Don't mix `loc` and `iloc`! This will cause an error:

In [ ]:
# ⚠️ INTENTIONAL ERROR - This will fail!
# You cannot use a string label with iloc
# df_reset.iloc[0, "fare"]
# Error: iloc cannot index with a string key

✅ **Key Takeaway:** Remember - `.loc[]` uses labels, `.iloc[]` uses positions. With `RangeIndex` they work the same, but this changes with custom indices!

---

## 4. Label-based index

📚 **Concept:** Instead of numbers, you can use meaningful labels as your index - like names, dates, or IDs. This makes it easier to look up specific rows.

<!-- INSTRUCTOR: Emphasize real-world use cases - customer IDs, product codes, timestamps, etc. -->

In [ ]:
# Load Olympic medals dataset
df = pd.read_csv("summer.csv")
df.head()

In [ ]:
# Currently has default RangeIndex
df.index

### 4.1 Setting a custom index

💡 **Tip:** Use `.set_index()` to choose which column should become the index.

In [ ]:
# Set Athlete names as the index
df = df.set_index("Athlete")
df.head()

In [ ]:
# Now we have a label-based Index
df.index

💡 **Tip:** You can also set the index when reading a CSV file:

```python
df = pd.read_csv("summer.csv", index_col="Athlete")
```

### 4.2 Using `loc` with label-based indices

📚 **Concept:** Now `.loc[]` and `.iloc[]` work differently! `.loc[]` uses the athlete names, while `.iloc[]` still uses position numbers.

In [ ]:
# Find all medals for a specific athlete using loc
df.loc["PHELPS, Michael"]

💡 **Tip:** When multiple rows share the same index label, `.loc[]` returns all of them as a DataFrame.

In [ ]:
# Select specific columns for an athlete
df.loc["PHELPS, Michael", ["Year", "Medal"]]

In [ ]:
# Select multiple athletes
df.loc[["PHELPS, Michael", "HAJOS, Alfred"], ["Year", "Medal"]]

✅ **Key Takeaway:** Label-based indices let you look up data by meaningful names instead of remembering position numbers.

### ✅ Quick Check
Before proceeding, can you:
- [ ] Explain the difference between `.loc[]` and `.iloc[]`?
- [ ] Set a column as the index using `.set_index()`?
- [ ] Reset an index back to 0, 1, 2... using `.reset_index()`?

---

## 5. MultiIndex (hierarchical indexing)

📚 **Concept:** A `MultiIndex` has multiple levels - like organizing data by category and subcategory. Think of it like a pivot table or nested folders.

**Example use cases:**
- Stock prices by (company, date)
- Sales data by (region, store, product)
- Student grades by (school, class, student)

<!-- INSTRUCTOR: Draw a visual hierarchy on the board to help students understand nesting -->

In [ ]:
# Load stock prices dataset
stocks = pd.read_csv("stocks_short.csv")
stocks

### 5.1 Creating a MultiIndex

💡 **Tip:** Pass a list of column names to `.set_index()` to create multiple index levels.

In [ ]:
# Set both Symbol and Date as index
stocks = stocks.set_index(["Symbol", "Date"])
stocks

In [ ]:
# Verify we have a MultiIndex
type(stocks.index)

In [ ]:
# See the hierarchical structure
stocks.index

📚 **Concept:** Each index position is a tuple (pair) of values:
- Level 0 (outer): Symbol (AAPL, CSCO, MSFT)
- Level 1 (inner): Date (2016-10-03, 2016-10-04, etc.)

In [ ]:
# First index is a tuple
stocks.index[0]

In [ ]:
# Access individual levels
print("Level 0 (Symbol):", stocks.index[0][0])
print("Level 1 (Date):", stocks.index[0][1])

### 5.2 Sorting with MultiIndex

💡 **Tip:** Always sort a `MultiIndex` for better performance and easier navigation. Sorting happens first by level 0, then by level 1.

In [ ]:
# Sort by index to group symbols together
stocks = stocks.sort_index()
stocks

### 5.3 Selecting data with MultiIndex

📚 **Concept:** You can select data at different levels:
1. Select by outer level only → returns all matching rows
2. Select by both levels → returns specific rows
3. Select multiple items using lists of tuples

In [ ]:
# Get all rows for AAPL (level 0 only)
stocks.loc["AAPL"]

In [ ]:
# Get AAPL on a specific date (both levels)
# Use a tuple: (level_0, level_1)
stocks.loc[("AAPL", "2016-10-03")]

In [ ]:
# Get multiple stocks on the same date
# Use a list of tuples
stocks.loc[
    [
        ("AAPL", "2016-10-03"),
        ("CSCO", "2016-10-03")
    ]
]

✅ **Key Takeaway:** `MultiIndex` organizes data hierarchically, letting you slice data at different levels of granularity.

---

## Summary

Congratulations! You now understand pandas index objects. Let's recap what you learned:

### Index Types
| Index Type | Description | Use Case |
|------------|-------------|----------|
| `RangeIndex` | Default 0, 1, 2... sequence | General purpose, when row order matters |
| `Index` | Custom labels (strings, dates, IDs) | When rows have unique identifiers |
| `MultiIndex` | Multiple hierarchical levels | Nested/grouped data (region > store > product) |

### Key Operations
| Operation | Method | Purpose |
|-----------|--------|---------|  
| Reset index | `.reset_index(drop=True)` | Create clean 0, 1, 2... sequence |
| Set custom index | `.set_index("column")` | Use column values as labels |
| Sort index | `.sort_index()` | Organize index alphabetically/numerically |
| Select by label | `.loc[label]` | Access rows by index label |
| Select by position | `.iloc[position]` | Access rows by numeric position |

### Selection Patterns
```python
# RangeIndex
df.loc[0]              # Row with label 0
df.iloc[0]             # First row (position 0)

# Label-based Index  
df.loc["John"]         # Row(s) labeled "John"
df.loc[["John", "Mary"], ["age", "city"]]  # Multiple rows & columns

# MultiIndex
df.loc["AAPL"]         # All rows for level 0 = AAPL
df.loc[("AAPL", "2020-01-01")]  # Specific tuple of levels
```

### ✅ Final Check
You should now be able to:
- [ ] Explain what an index is and why it's useful
- [ ] Distinguish between `RangeIndex`, `Index`, and `MultiIndex`
- [ ] Use `.set_index()` and `.reset_index()` appropriately
- [ ] Choose between `.loc[]` and `.iloc[]` for data selection
- [ ] Navigate hierarchical data with `MultiIndex`
- [ ] Apply modern pandas patterns (method chaining over `inplace=True`)

Great work! You're ready to practice these concepts in the lab exercises.

---

## Additional Resources

- [Pandas Index Objects Documentation](https://pandas.pydata.org/docs/reference/api/pandas.Index.html)
- [MultiIndex Documentation](https://pandas.pydata.org/docs/user_guide/advanced.html)
- Practice with the lab exercises to reinforce these concepts!